# Top1-Top2 Token Farkı — Veri Toplama
**Amaç:** cosmos-T1 modelini gsm8k_tr veri seti üzerinde çalıştırarak her token için top1-top2 logit farkını kaydetmek.

In [1]:
# ── Hücre 1: Kurulum & Veri Seti ──────────────────────────────────────────────
!pip install -q transformers datasets accelerate

from datasets import load_dataset
import re

# Veri setini çek
ds = load_dataset("ytu-ce-cosmos/gsm8k_tr")
train_ds = ds["train"]

print(f"Train: {len(train_ds)} soru")
print(f"Kolonlar: {train_ds.column_names}")
print("\n" + "="*60)
print("3-5 Örnek Soru-Cevap:")
print("="*60)

for i in range(5):
    row = train_ds[i]
    # Kolon isimlerini otomatik tespit et
    q_col = [c for c in train_ds.column_names if 'question' in c.lower() or 'soru' in c.lower()][0]
    a_col = [c for c in train_ds.column_names if 'answer' in c.lower() or 'cevap' in c.lower()][0]
    print(f"\n[{i}] SORU  : {row[q_col]}")
    print(f"    CEVAP : {row[a_col]}")

Train: 8792 soru
Kolonlar: ['question', 'answer']

3-5 Örnek Soru-Cevap:

[0] SORU  : Borris tekel bayisi her 6 ayda bir 90 kilogram üzüm kullanıyor. Üretimini yüzde yirmi oranında artırmayı düşünüyor. Üretimini artırdıktan sonra bir yıl içinde kaç üzüme ihtiyacı olur?
    CEVAP : Borris şu anda her 6 ayda 90 kilogram üzüm kullanıyorsa, bir yılda 180 kilogram üzüm kullanıyor demektir. Üretimini yüzde 20 oranında artırırsa:

180 kilogram x 1.20 = 216 kilogram

Buna göre, üretimini artırdıktan sonra bir yıl içinde 216 kilogram üzüme ihtiyacı olacaktır.

[1] SORU  : Mel, Katherine'den üç yaş küçük.  Katherine iki düzine yaşına geldiğinde Mel kaç yaşında olacak?
    CEVAP : Katherine iki düzine yaşına geldiğinde 24 yaşında olacaktır. Mel ise Katherine'den üç yaş küçük olduğuna göre, Katherine 24 yaşındayken Mel 21 yaşında olacaktır.

[2] SORU  : James 2 ağacındaki tüm meyveleri toplar.  Her ağaçta 20 bitki var.  Her bitkinin 1 tohumu var ve bunların %60'ını ekiyor.  Kaç ağaç dikmiştir?
   

In [2]:
# ── Hücre 2: Model Yükleme & GPU Kontrolü ─────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "ytu-ce-cosmos/Turkish-Gemma-9b-T1"  # cosmos-T1
# NOT: Hugging Face'deki gerçek model adını buraya yazın.

print("Model yükleniyor:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# GPU durumu
if torch.cuda.is_available():
    print(f"\n✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB toplam")
    print(f"  Kullanılan: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    print("⚠ GPU bulunamadı, CPU kullanılıyor.")

print(f"\nModel cihaz haritası: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'tek cihaz'}")

Model yükleniyor: ytu-ce-cosmos/Turkish-Gemma-9b-T1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]


✓ GPU: NVIDIA A100-SXM4-80GB
  Bellek: 85.1 GB toplam
  Kullanılan: 18.48 GB

Model cihaz haritası: tek cihaz


In [3]:
import torch.nn.functional as F
import re
def extract_number(text):
    """
    Önce 'Cevap:', '####', '\boxed{}' gibi kalıpları arar.
    Bulamazsa son sayısal değeri alır.
    Virgüllü Türkçe ondalık sayıları doğru işler.
    """
    if not text:
        return None

    # \boxed{42} formatı
    match = re.search(r'\\boxed\{(-?[\d.,]+)\}', text)
    if match:
        return _parse_num(match.group(1))

    # #### 42 formatı
    match = re.search(r'####\s*(-?[\d.,]+)', text)
    if match:
        return _parse_num(match.group(1))

    # "Cevap: 42" veya "cevap 42" formatı
    match = re.search(r'[Cc]evap\s*:?\s*(-?[\d.,]+)', text)
    if match:
        return _parse_num(match.group(1))

    # "= 42" formatı (son eşittirden sonraki sayı)
    matches = re.findall(r'=\s*(-?[\d.,]+)\s*(?:$|\n|kg|km|dolar|\$|TL|saat|yıl)', text)
    if matches:
        return _parse_num(matches[-1])

    # Son çare: metindeki son tam sayı (ondalık değil)
    matches = re.findall(r'\b(-?\d+)\b', text)
    if matches:
        return _parse_num(matches[-1])

    return None

def _parse_num(s):
    """'1.234,56' veya '1,234.56' veya '42' → float"""
    s = s.strip()
    # Türkçe format: nokta binlik ayraç, virgül ondalık → 1.234,56
    if re.match(r'^\d{1,3}(\.\d{3})*(,\d+)?$', s):
        s = s.replace('.', '').replace(',', '.')
    else:
        # İngilizce format ya da sadece virgüllü: 1,234 → 1234
        s = s.replace(',', '')
    try:
        return float(s)
    except:
        return None

def run_and_collect(question_text, question_id, q_col, a_col, row=None):
    """
    Soruyu modele sorar, her token için top1-top2 farkını hesaplar.
    Döndürür: dict (question_id, top1_top2_list, prediction, ground_truth, is_correct)
    """
    # Prompt hazırla
    prompt = f"Soru: {question_text}\nCevap:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    think_end_id = tokenizer.encode("</think>", add_special_tokens=False)

    with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=512,
          do_sample=False,
          eos_token_id=[tokenizer.eos_token_id] + think_end_id,
          return_dict_in_generate=True,
          output_scores=True,
          pad_token_id=tokenizer.eos_token_id
      )

    # Üretilen token'lar (sadece yeni kısım)
    generated_ids = outputs.sequences[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Her adım için top1-top2 farkı
    top1_top2_list = []
    for step_scores in outputs.scores:
        probs = F.softmax(step_scores[0], dim=-1)  # (vocab_size,)
        top2  = torch.topk(probs, 2)
        diff  = (top2.values[0] - top2.values[1]).item()
        top1_top2_list.append(round(diff, 6))

    prediction   = extract_number(generated_text)
    ground_truth = extract_number(row[a_col]) if row else None

    if prediction is not None and ground_truth is not None:
        is_correct = abs(prediction - ground_truth) < 1e-3
    else:
        is_correct = False

    return {
        "question_id"    : question_id,
        "generated_text"  : generated_text,
        "top1_top2_list" : top1_top2_list,
        "prediction"     : prediction,
        "ground_truth"   : ground_truth,
        "is_correct"     : bool(is_correct),
    }

In [ ]:
# ── 10 soru ile deneme ──────────────────────────────────────────────────────────
q_col = [c for c in train_ds.column_names if 'question' in c.lower() or 'soru' in c.lower()][0]
a_col = [c for c in train_ds.column_names if 'answer' in c.lower() or 'cevap' in c.lower()][0]

for test_idx in range(10):
    row = train_ds[test_idx]
    result = run_and_collect(row[q_col], test_idx, q_col, a_col, row)

    print(f"\n{'='*65}")
    print(f"[Soru #{test_idx}]")
    print(f"  Tam soru  : {row[q_col]}")
    print(f"  Tam cevap  : {result['generated_text']}")

    print(f"  Top1-Top2 farkları ({len(result['top1_top2_list'])} token):")
    print(f"    {result['top1_top2_list']} ...")
    print(f"  Tahmin (sayısal): {result['prediction']}")
    print(f"  Ground Truth    : {result['ground_truth']}")
    print(f"  is_correct      : {result['is_correct']}")


[Soru #0]
  Tam soru  : Borris tekel bayisi her 6 ayda bir 90 kilogram üzüm kullanıyor. Üretimini yüzde yirmi oranında artırmayı düşünüyor. Üretimini artırdıktan sonra bir yıl içinde kaç üzüme ihtiyacı olur?
  Tam cevap  :  Öncelikle, Borris'in şu anki üzüm tüketimini yıllık olarak hesaplayalım. Her 6 ayda bir 90 kg üzüm kullanıyorsa, bir yılda iki kez bu miktarı kullanır. Yani yıllık tüketim: 2 * 90 kg = 180 kg.

Şimdi, üretimi %20 artırmayı planlıyor. Bu, üzüm kullanımının da %20 artacağı anlamına gelir. Yani yeni yıllık tüketim: 180 kg * 1.20 = 216 kg.

Soruda, "bir yıl içinde kaç üzüme ihtiyacı olur?" diye soruluyor. Ancak, kg cinsinden hesapladık, üzüme çevirmemiz gerekiyor. Burada bir eksiklik var. Üzümlerin ağırlığına dair bir bilgi yok. Soruda üzümün ağırlığı verilmemiş. Bu nasıl çözülecek?

Belki de üzümlerin sayısını hesaplamak için bir varsayım yapmamız gerekiyor. Örneğin, ortalama bir üzümün ağırlığını bilmemiz lazım. Ama soruda bu bilgi yok.

Tekrar okuyorum: "Borris teke

In [4]:
import pickle
import json
import time
from tqdm.auto import tqdm

N_START = 0    # Part 2 için 600 yap
N_END   = 600  # Part 2 için 1200 yap
PART    = 1    # Part 2 için 2 yap

n_use = min(N_END, len(train_ds))
print(f"İşlenecek sorular: {N_START} - {n_use}")

results = []
errors  = []
start_time = time.time()

for idx in tqdm(range(N_START, n_use), desc="İşleniyor", unit="soru"):
    row = train_ds[idx]
    try:
        info = run_and_collect(row[q_col], idx, q_col, a_col, row)
        if "</think>" in info["generated_text"]:
            info["generated_text"] = info["generated_text"].split("</think>")[0]
        results.append(info)
    except Exception as e:
        errors.append({"idx": idx, "error": str(e)})

    if (len(results) + len(errors)) % 100 == 0:
        elapsed     = time.time() - start_time
        done_so_far = len(results) + len(errors)
        remaining   = (n_use - N_START - done_so_far) * (elapsed / max(done_so_far, 1))
        correct     = sum(r["is_correct"] for r in results)
        print(f"  [{done_so_far}/{n_use - N_START}] "
              f"Doğru: {correct}/{len(results)} ({100*correct/max(len(results),1):.1f}%) | "
              f"Kalan: ~{remaining/60:.1f} dk")
        with open(f"samples_infos_part{PART}_checkpoint.pkl", "wb") as f:
            pickle.dump({"samples": results, "errors": errors}, f)

# Son kayıt ve indirme
total_time    = time.time() - start_time
correct_total = sum(r["is_correct"] for r in results)

with open(f"samples_infos_part{PART}.pkl", "wb") as f:
    pickle.dump({
        "meta": {
            "model"      : MODEL_NAME,
            "part"       : PART,
            "n_start"    : N_START,
            "n_end"      : n_use,
            "n_total"    : len(results),
            "accuracy"   : round(correct_total / max(len(results), 1), 4),
            "elapsed_min": round(total_time / 60, 2)
        },
        "samples": results,
        "errors" : errors
    }, f)

print(f"\n✓ Tamamlandı: {len(results)} soru | "
      f"Doğru: {correct_total} ({100*correct_total/max(len(results),1):.1f}%) | "
      f"Süre: {total_time/60:.1f} dk")

from google.colab import files
files.download(f"samples_infos_part{PART}.pkl")

İşlenecek sorular: 0 - 600


İşleniyor:   0%|          | 0/600 [00:00<?, ?soru/s]

  [100/600] Doğru: 45/100 (45.0%) | Kalan: ~287.1 dk
  [200/600] Doğru: 83/200 (41.5%) | Kalan: ~226.0 dk
  [300/600] Doğru: 122/300 (40.7%) | Kalan: ~169.6 dk
  [400/600] Doğru: 176/400 (44.0%) | Kalan: ~112.2 dk
  [500/600] Doğru: 222/500 (44.4%) | Kalan: ~55.9 dk
  [600/600] Doğru: 274/600 (45.7%) | Kalan: ~0.0 dk

✓ Tamamlandı: 600 soru | Doğru: 274 (45.7%) | Süre: 335.0 dk


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import pickle
import json
import time
from tqdm.auto import tqdm

N_START = 600
N_END   = 1200
PART    = 2

n_use = min(N_END, len(train_ds))
print(f"İşlenecek sorular: {N_START} - {n_use}")

results = []
errors  = []
start_time = time.time()

for idx in tqdm(range(N_START, n_use), desc="İşleniyor", unit="soru"):
    row = train_ds[idx]
    try:
        info = run_and_collect(row[q_col], idx, q_col, a_col, row)
        if "</think>" in info["generated_text"]:
            info["generated_text"] = info["generated_text"].split("</think>")[0]
        results.append(info)
    except Exception as e:
        errors.append({"idx": idx, "error": str(e)})

    if (len(results) + len(errors)) % 100 == 0:
        elapsed     = time.time() - start_time
        done_so_far = len(results) + len(errors)
        remaining   = (n_use - N_START - done_so_far) * (elapsed / max(done_so_far, 1))
        correct     = sum(r["is_correct"] for r in results)
        print(f"  [{done_so_far}/{n_use - N_START}] "
              f"Doğru: {correct}/{len(results)} ({100*correct/max(len(results),1):.1f}%) | "
              f"Kalan: ~{remaining/60:.1f} dk")
        with open(f"samples_infos_part{PART}_checkpoint.pkl", "wb") as f:
            pickle.dump({"samples": results, "errors": errors}, f)

# Son kayıt ve indirme
total_time    = time.time() - start_time
correct_total = sum(r["is_correct"] for r in results)

with open(f"samples_infos_part{PART}.pkl", "wb") as f:
    pickle.dump({
        "meta": {
            "model"      : MODEL_NAME,
            "part"       : PART,
            "n_start"    : N_START,
            "n_end"      : n_use,
            "n_total"    : len(results),
            "accuracy"   : round(correct_total / max(len(results), 1), 4),
            "elapsed_min": round(total_time / 60, 2)
        },
        "samples": results,
        "errors" : errors
    }, f)

print(f"\n✓ Tamamlandı: {len(results)} soru | "
      f"Doğru: {correct_total} ({100*correct_total/max(len(results),1):.1f}%) | "
      f"Süre: {total_time/60:.1f} dk")

from google.colab import files
files.download(f"samples_infos_part{PART}.pkl")

İşlenecek sorular: 600 - 1200


İşleniyor:   0%|          | 0/600 [00:00<?, ?soru/s]

  [100/600] Doğru: 43/100 (43.0%) | Kalan: ~268.7 dk
  [200/600] Doğru: 85/200 (42.5%) | Kalan: ~216.0 dk
  [300/600] Doğru: 133/300 (44.3%) | Kalan: ~161.5 dk
  [400/600] Doğru: 182/400 (45.5%) | Kalan: ~107.2 dk
  [500/600] Doğru: 226/500 (45.2%) | Kalan: ~53.7 dk
  [600/600] Doğru: 275/600 (45.8%) | Kalan: ~0.0 dk

✓ Tamamlandı: 600 soru | Doğru: 275 (45.8%) | Süre: 322.0 dk


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>